# StressID and Experiment Dataset comparison

In [1]:
%matplotlib widget
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
from IPython.display import display

from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.feature_selection import RFECV
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold

STRESSID_DATA_PATH = "../../.."
STRESSID_LABELS_SEPARATOR = ","
STRESSID_LABELS_FILENAME = f"{STRESSID_DATA_PATH}/stressID/labels.csv"
STRESSID_FEATURES_SEPARATOR = ","
STRESSID_FEATURES_DIRECTORY = f"{STRESSID_DATA_PATH}/Reprod-Features-NK"
STRESSID_FEATURES_FILENAME = f"{STRESSID_FEATURES_DIRECTORY}/ecg_eda_features.csv"
STRESSID_RAWDATA_PATH = f"{STRESSID_DATA_PATH}/stressid-data"

EXPDATA_DATA_PATH = "../../../experiment-data"
EXPDATA_LABELS_SEPARATOR = ","
EXPDATA_LABELS_FILENAME = f"{EXPDATA_DATA_PATH}/labels.csv"
EXPDATA_FEATURES_SEPARATOR = ";"
EXPDATA_FEATURES_DIRECTORY = f"{EXPDATA_DATA_PATH}/extracted-features"
EXPDATA_FEATURES_FILENAME = f"{EXPDATA_FEATURES_DIRECTORY}/all_features.csv"

In [2]:
stressid_df = pd.read_csv(STRESSID_FEATURES_FILENAME, sep=STRESSID_FEATURES_SEPARATOR, index_col=0)
n_rows, n_cols = stressid_df.shape
print("======================================================")
print(f"StressID dataset contains {n_rows} samples with {n_cols} features")
# display(stressid_df)

si_labels_df = pd.read_csv(STRESSID_LABELS_FILENAME, sep=STRESSID_LABELS_SEPARATOR, index_col=0)
n_rows, n_cols = si_labels_df.shape
print("======================================================")
print(f"StressID labels contains {n_rows} labels with {n_cols} types of classifications")
# display(si_labels_df)

# Selecting rows that actually have entries both labels and samples
idx = list(stressid_df.merge(si_labels_df, left_index=True, right_index=True).index)
si_labels = si_labels_df.loc[idx]
si_X = stressid_df.loc[idx]
n_rows, n_cols = si_X.shape
print("======================================================")
print(f"StressID classification dataset contains {n_rows} samples with {n_cols} features")

si_bclass_labels = si_labels["binary-stress"]
# display(bclass_labels)


expdata_df = pd.read_csv(EXPDATA_FEATURES_FILENAME, sep=EXPDATA_FEATURES_SEPARATOR, index_col=0)
n_rows, n_cols = expdata_df.shape
print("\n\n======================================================")
print(f"ExpData dataset contains {n_rows} samples with {n_cols} features")
# display(expdata_df)

ed_labels_df = pd.read_csv(EXPDATA_LABELS_FILENAME, sep=EXPDATA_LABELS_SEPARATOR, index_col=0)
n_rows, n_cols = ed_labels_df.shape
print("======================================================")
print(f"ExpData labels contains {n_rows} labels with {n_cols} types of classifications")
# display(ed_labels_df)

# Selecting rows that actually have entries both labels and samples
idx = list(expdata_df.merge(ed_labels_df, left_index=True, right_index=True).index)
ed_labels = ed_labels_df.loc[idx]
ed_X = expdata_df.loc[idx]
n_rows, n_cols = ed_X.shape
print("======================================================")
print(f"ExpData classification dataset contains {n_rows} samples with {n_cols} features")

ed_bclass_labels = ed_labels["binary-stress"]
# display(bclass_labels)

StressID dataset contains 773 samples with 70 features
StressID labels contains 700 labels with 3 types of classifications
StressID classification dataset contains 699 samples with 70 features


ExpData dataset contains 147 samples with 70 features
ExpData labels contains 126 labels with 3 types of classifications
ExpData classification dataset contains 126 samples with 70 features


### Dimension/Feature reduction

In [3]:
# PCA
STD_EXPL_RATIO = 0.95

si_pca_95p = PCA(n_components=STD_EXPL_RATIO, svd_solver="full")
scaled = StandardScaler().fit_transform(si_X)
si_X_95p = si_pca_95p.fit_transform(scaled)

n_rows, n_cols = si_X_95p.shape
ratios = [(f"{x * 100:.2f}") for x in si_pca_95p.explained_variance_ratio_]
print(f"StressID classification dataset reduced to {n_cols} derived features that explains >{STD_EXPL_RATIO*100}% of variance in the dataset")
print(f"Features contribution ratio to variance: {ratios}")



ed_pca_95p = PCA(n_components=STD_EXPL_RATIO, svd_solver="full")
scaled = StandardScaler().fit_transform(ed_X)
ed_X_95p = ed_pca_95p.fit_transform(scaled)

n_rows, n_cols = ed_X_95p.shape
ratios = [(f"{x * 100:.2f}") for x in ed_pca_95p.explained_variance_ratio_]
print(
    f"\n\nExpData classification dataset reduced to {n_cols} derived features that explains >{STD_EXPL_RATIO * 100}% of variance in the dataset"
)
print(f"Features contribution ratio to variance: {ratios}")

StressID classification dataset reduced to 26 derived features that explains >95.0% of variance in the dataset
Features contribution ratio to variance: ['19.51', '12.45', '9.35', '7.55', '6.15', '5.42', '4.47', '3.83', '3.16', '2.71', '2.31', '1.97', '1.91', '1.78', '1.46', '1.41', '1.32', '1.28', '1.13', '1.03', '0.93', '0.89', '0.85', '0.78', '0.69', '0.66']


ExpData classification dataset reduced to 22 derived features that explains >95.0% of variance in the dataset
Features contribution ratio to variance: ['25.23', '12.55', '9.76', '7.94', '6.48', '5.07', '4.07', '3.48', '2.99', '2.63', '2.16', '2.00', '1.67', '1.54', '1.43', '1.33', '1.07', '1.01', '0.92', '0.81', '0.70', '0.64']


In [4]:
SPLITS = 10
RAN_STATE = 21

In [5]:
# RFECV for StressID
estimator = RandomForestClassifier(max_depth=3, random_state=RAN_STATE)
cv = StratifiedKFold(n_splits=SPLITS, shuffle=True, random_state=RAN_STATE)
selector = RFECV(estimator=estimator, step=2, cv=cv, scoring="balanced_accuracy")
selector.fit(si_X, si_bclass_labels)
features_mask = selector.support_
si_X_rfe = si_X.loc[:, features_mask]

features_scores = { "scores": [], "features": [] }
for score, feat in zip(selector.estimator_.feature_importances_, si_X_rfe.columns):
    features_scores["scores"].append(score)
    features_scores["features"].append(feat)
features_scores_df = pd.DataFrame(features_scores).sort_values(by="scores", ascending=False)
features_scores_df = features_scores_df.reset_index(drop=True)
features_scores_df.index = features_scores_df.index + 1
print("Selected features scores for StressID")
display(features_scores_df)

Selected features scores for StressID


,scores,features
1,0.238156,pNN20
2,0.185845,ULF
3,0.117940,sampEn
4,0.086595,pNN50
5,0.049868,totalpower
6,0.041830,apEn
7,0.035932,q1_ecg
8,0.034558,sd_eda
9,0.034300,peakLF
10,0.030622,CVNN


In [6]:
# RFECV for ExpData
estimator = RandomForestClassifier(max_depth=3, random_state=RAN_STATE)
cv = StratifiedKFold(n_splits=SPLITS, shuffle=True, random_state=RAN_STATE)
selector = RFECV(estimator=estimator, step=2, cv=cv, scoring="balanced_accuracy")
selector.fit(ed_X, ed_bclass_labels)
features_mask = selector.support_
ed_X_rfe = ed_X.loc[:, features_mask]

features_scores = {"scores": [], "features": []}
for score, feat in zip(selector.estimator_.feature_importances_, ed_X_rfe.columns):
    features_scores["scores"].append(score)
    features_scores["features"].append(feat)
features_scores_df = pd.DataFrame(features_scores).sort_values(by="scores", ascending=False)
features_scores_df = features_scores_df.reset_index(drop=True)
features_scores_df.index = features_scores_df.index + 1
print("Selected features scores for ExpData")
display(features_scores_df)

Selected features scores for ExpData


,scores,features
1,0.210688,sumAmpSCR
2,0.177161,sd_scl
3,0.168773,sdHR
4,0.157930,SDSD
5,0.154041,pNN20
6,0.131407,median_ecg


### t-SNE

In [ ]:
# Divergence analysis on StressID
perplexity = np.arange(30, 390, 30)
divergence = []
si_Ncomp = 2

for i in perplexity:
    model = TSNE(n_components=si_Ncomp, init="pca", perplexity=i)
    reduced = model.fit_transform(si_X)
    divergence.append(model.kl_divergence_)

fig = px.line(x=perplexity, y=divergence, markers=True, width=800, height=600)
fig.update_layout(xaxis_title="Perplexity Values", yaxis_title="KL Divergence")
fig.update_traces(line_color="red", line_width=1)
fig.show()

In [29]:
# t-SNE in StressID
# Best N-comp=2, Perp=300 np.arange(15, 330, 15)
si_Ncomp = 2
si_Perp = 300

si_tsne = TSNE(n_components=si_Ncomp, perplexity=si_Perp, random_state=RAN_STATE)
si_X_tsne = si_tsne.fit_transform(si_X)
display(si_tsne.kl_divergence_)

fig = px.scatter(x=si_X_tsne[:,0], y=si_X_tsne[:,1], color=si_bclass_labels, width=800, height=600)
fig.update_layout(
    title="t-SNE visualization of StressID dataset",
    xaxis_title="1st t-SNE",
    yaxis_title="2nd t-SNE",
)
fig.show()

0.04706011340022087

In [43]:
# t-SNE in StressID
# Best N-comp=3, Perp=360 np.arange(30, 390, 30)
si_Ncomp = 3
si_Perp = 360

si_tsne = TSNE(n_components=si_Ncomp, perplexity=si_Perp, random_state=RAN_STATE)
si_X_tsne = si_tsne.fit_transform(si_X)
display(si_tsne.kl_divergence_)

fig = px.scatter_3d(x=si_X_tsne[:, 0], y=si_X_tsne[:, 1], z=si_X_tsne[:,2], color=si_bclass_labels, opacity=0.7, width=800, height=600)
fig.update_layout(
    title="t-SNE visualization of StressID dataset",
    xaxis_title="1st t-SNE",
    yaxis_title="2nd t-SNE",
)
fig.show()


0.024578262120485306

In [57]:
# Divergence analysis on StressID
perplexity = np.arange(25, 700, 25)
divergence = []
si_Ncomp = 3

for i in perplexity:
    model = TSNE(n_components=si_Ncomp, init="pca", perplexity=i)
    reduced = model.fit_transform(si_X_rfe)
    divergence.append(model.kl_divergence_)

fig = px.line(x=perplexity, y=divergence, markers=True, width=800, height=600)
fig.update_layout(xaxis_title="Perplexity Values", yaxis_title="KL Divergence")
fig.update_traces(line_color="red", line_width=1)
fig.show()


In [77]:
# t-SNE in StressID with PCA si_X_95p
# Best N-comp=2, Perp=300 np.arange(50, 700, 50)
# Best N-comp=3, Perp from 550 to 700, np.arange(50, 700, 50)

# t-SNE in StressID with RFE si_X_rfe
# Best N-comp=2, Perp=225 onwards np.arange(25, 700, 25)
# Best N-comp=3, Perp=150 onwards np.arange(25, 700, 25)

si_Ncomp = 3
si_Perp = 250

si_tsne = TSNE(n_components=si_Ncomp, perplexity=si_Perp, random_state=RAN_STATE)
si_X_tsne = si_tsne.fit_transform(si_X_rfe)
display(si_tsne.kl_divergence_)

#fig = px.scatter(x=si_X_tsne[:, 0], y=si_X_tsne[:, 1], color=si_bclass_labels, width=800, height=600)
fig = px.scatter_3d(
    x=si_X_tsne[:, 0], y=si_X_tsne[:, 1], z=si_X_tsne[:, 2], color=si_bclass_labels, opacity=0.7, width=800, height=600
)
fig.update_layout(
    title="t-SNE visualization of StressID dataset",
    xaxis_title="1st t-SNE",
    yaxis_title="2nd t-SNE",
)
fig.show()


0.02303946763277054

In [84]:
# Divergence analysis on ExpData
perplexity = np.arange(5, 100, 5)
divergence = []
ed_Ncomp = 3

for i in perplexity:
    model = TSNE(n_components=ed_Ncomp, perplexity=i, learning_rate=30)
    reduced = model.fit_transform(ed_X_rfe)
    divergence.append(model.kl_divergence_)

fig = px.line(x=perplexity, y=divergence, markers=True, width=800, height=600)
fig.update_layout(xaxis_title="Perplexity Values", yaxis_title="KL Divergence")
fig.update_traces(line_color="red", line_width=1)
fig.show()


In [98]:
# t-SNE in StressID with RFE
# Best N-comp=2, Perp=50 onwards (perp range(5, 100, 5))
# Best N-comp=3, Perp=35 onwards (perp range(5, 100, 5)) learning_rate = 30
ed_Ncomp = 3
ed_Perp = 60

ed_tsne = TSNE(n_components=ed_Ncomp, perplexity=ed_Perp, random_state=RAN_STATE, learning_rate=30)
ed_X_tsne = ed_tsne.fit_transform(ed_X_rfe)
display(ed_tsne.kl_divergence_)

#fig = px.scatter(x=ed_X_tsne[:,0], y=ed_X_tsne[:,1], color=ed_bclass_labels, width=800, height=600)
fig = px.scatter_3d(
    x=si_X_tsne[:, 0], y=si_X_tsne[:, 1], z=si_X_tsne[:, 2], color=si_bclass_labels, opacity=0.7, width=800, height=600
)
fig.update_layout(
    title="t-SNE visualization of StressID dataset",
    xaxis_title="1st t-SNE",
    yaxis_title="2nd t-SNE",
)
fig.show()


0.012605981901288033

In [13]:
# t-SNE in StressID
# Best N-comp=3, Perp=10 (perp range(1, 40, 1))
ed_Ncomp = 3
ed_Perp = 10

ed_tsne = TSNE(n_components=ed_Ncomp, perplexity=ed_Perp, random_state=RAN_STATE)
ed_X_tsne = ed_tsne.fit_transform(ed_X)
display(ed_tsne.kl_divergence_)

fig = px.scatter_3d(
    x=ed_X_tsne[:, 0], y=ed_X_tsne[:, 1], z=ed_X_tsne[:, 2], color=ed_bclass_labels, opacity=0.7, width=800, height=600
)
fig.update_layout(
    title="t-SNE visualization of StressID dataset",
    xaxis_title="1st t-SNE",
    yaxis_title="2nd t-SNE",
)
fig.show()

0.18972700834274292